In [ ]:
# CELL 1 - PDF EXPORT SETUP
from IPython.display import HTML, display
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
display(HTML('<style>.container{width:100%!important}.output{overflow-x:auto}</style>'))
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['savefig.bbox'] = 'tight'
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.max_colwidth', 60)
print('Setup complete. Run cells top to bottom.')


# India Pharma Supply Chain Intelligence System
## CIIP Index v2.0 - Upstream Risk Scoring and Demand Forecasting

**Developer:** Aditya V Sivaram Poduri | India Supply Chain Signals

**Architecture:**
- Layer 1: Prophet forecasting for 5 pharma upstream inputs
- Layer 2: CIIP Index cross-industry risk scoring
- Layer 3: MAPE/MAE/RMSE + Monte Carlo + Scenario Engine + Confidence Scoring

**Stack:** Python | Prophet | Pandas | NumPy | Plotly | Streamlit | scikit-learn


In [ ]:
# CELL 3 - INSTALL DEPENDENCIES
!pip install prophet pandas matplotlib plotly scikit-learn streamlit -q
print('All dependencies installed.')


In [ ]:
# CELL 4 - LOAD AND VALIDATE ALL DATASETS
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

material_files = {
    'Soda Ash':    'Soda_Ash_2015_2025.csv',
    'Methanol':    'Methanol_2015_2025.csv',
    'Acetic Acid': 'Acetic_Acid_2015_2025.csv',
    'LNG Japan':   'LNG_Japan_2015_2025.csv',
    'Polysilicon': 'Polysilicon_2015_2025.csv'
}

print('=' * 65)
print('DATASET VALIDATION REPORT')
print('=' * 65)
datasets = {}
for material, filename in material_files.items():
    df = pd.read_csv(filename)
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').reset_index(drop=True)
    datasets[material] = df
    unit = df['Unit'].iloc[0] if 'Unit' in df.columns else 'N/A'
    print(f'{material}: {len(df)} rows | Unit: {unit} | Last price: {df["Price"].iloc[-1]:.2f}')
print('\nAll datasets loaded successfully.')


In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# INDIA PHARMA SUPPLY CHAIN INTELLIGENCE SYSTEM
# CIIP Index — Upstream Risk Scoring + Demand Forecasting
# Aditya V Sivaram Poduri | India Supply Chain Signals
# ============================================================

# --- LOAD ALL DATASETS ---
materials = {
    'Soda Ash': 'Soda_Ash_2015_2025.csv',
    'Methanol': 'Methanol_2015_2025.csv',
    'Acetic Acid': 'Acetic_Acid_2015_2025.csv',
    'LNG Japan': 'LNG_Japan_2015_2025.csv',
    'Polysilicon': 'Polysilicon_2015_2025.csv'
}

forecasts = {}

for material, filename in materials.items():
    df = pd.read_csv(filename)
    df = df.rename(columns={'Date': 'ds', 'Price': 'y'})
    df['ds'] = pd.to_datetime(df['ds'])

    # Use monthly frequency to avoid spike artifacts
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.05,
        seasonality_prior_scale=10
    )
    model.fit(df)

    # Forecast 12 months forward using monthly frequency
    future = model.make_future_dataframe(periods=12, freq='MS')
    forecast = model.predict(future)

    forecasts[material] = {
        'model': model,
        'forecast': forecast,
        'actual': df
    }

    print(f"{material}: Model trained. Last price: {df['y'].iloc[-1]:.2f} | 12m forecast: {forecast['yhat'].iloc[-1]:.2f}")

print("\nAll models trained successfully.")

In [ ]:
# --- PLOT ALL FIVE FORECASTS ---
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle('India Supply Chain Intelligence System\nCIIP Index — Upstream Input Price Forecasts (12-Month Forward)',
             fontsize=14, fontweight='bold', y=0.98)

axes_flat = axes.flatten()
colors = ['#1f4e79', '#c55a11', '#375623', '#7030a0', '#c00000']

for idx, (material, data) in enumerate(forecasts.items()):
    ax = axes_flat[idx]
    forecast = data['forecast']
    actual = data['actual']

    # Plot actual
    ax.plot(actual['ds'], actual['y'], 'k.', markersize=4, label='Actual', alpha=0.7)

    # Plot forecast
    ax.plot(forecast['ds'], forecast['yhat'], color=colors[idx], linewidth=2, label='Forecast')

    # Confidence interval
    ax.fill_between(forecast['ds'],
                    forecast['yhat_lower'],
                    forecast['yhat_upper'],
                    alpha=0.15, color=colors[idx])

    # Add vertical line at forecast start
    last_actual_date = actual['ds'].max()
    ax.axvline(x=last_actual_date, color='gray', linestyle='--', alpha=0.5, label='Forecast Start')

    ax.set_title(f'{material}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Date')
    unit = data['actual'].iloc[0]['Unit'] if 'Unit' in data['actual'].columns else ''

    # Get unit from original CSV
    orig = pd.read_csv(materials[material])
    unit = orig['Unit'].iloc[0]
    ax.set_ylabel(f'Price ({unit})')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# Hide the 6th subplot
axes_flat[5].set_visible(False)

plt.tight_layout()
plt.savefig(
    'CIIP_All_Forecasts.png',
    dpi=150,
    bbox_inches='tight',
    pad_inches=0.5
)
plt.tight_layout()
plt.show()
print("Forecast chart saved as CIIP_All_Forecasts.png")

In [ ]:
# --- CIIP INDEX SCORING ENGINE ---

def calculate_ciip_score(pharma, solar, construction, fmcg, geopolitical):
    """
    CIIP Index — Cross-Industry Input Pressure Score
    Developed by Aditya V Sivaram Poduri
    India Supply Chain Signals | indiasupplychainsignals.substack.com

    Weights:
    - Pharma demand pressure: 30%
    - Cross-industry competition (solar + construction + FMCG): 45%
    - Geopolitical supply risk: 25%

    Input scale: 1 (Low) to 3 (Critical)
    Output scale: 0 to 10
    """
    cross_industry = (solar + construction + fmcg) / 3
    raw_score = (pharma * 0.30 +
                 cross_industry * 0.45 +
                 geopolitical * 0.25)
    return round(raw_score * (10/3), 2)

# --- CIIP SCORES FOR EACH MATERIAL ---
ciip_data = {
    'Material': ['Soda Ash', 'Methanol', 'Acetic Acid', 'LNG Japan', 'Polysilicon'],
    'Pharma_Demand': [3, 3, 3, 3, 1],
    'Solar_Demand':  [3, 1, 1, 3, 3],
    'Construction':  [3, 2, 1, 3, 1],
    'FMCG_Demand':   [2, 1, 2, 2, 1],
    'Geopolitical':  [2, 3, 3, 3, 3],
}

ciip_df = pd.DataFrame(ciip_data)

ciip_df['CIIP_Score'] = ciip_df.apply(
    lambda row: calculate_ciip_score(
        row['Pharma_Demand'],
        row['Solar_Demand'],
        row['Construction'],
        row['FMCG_Demand'],
        row['Geopolitical']
    ), axis=1
)

# Add risk classification
def classify_risk(score):
    if score >= 8.0:
        return 'CRITICAL'
    elif score >= 6.5:
        return 'HIGH'
    elif score >= 5.0:
        return 'MEDIUM'
    else:
        return 'LOW'

ciip_df['Risk_Level'] = ciip_df['CIIP_Score'].apply(classify_risk)

# Add 12-month forecast direction
forecast_directions = []
for material in ciip_df['Material']:
    if material in forecasts:
        f = forecasts[material]['forecast']
        actual = forecasts[material]['actual']
        last_actual = actual['y'].iloc[-1]
        forecast_end = f['yhat'].iloc[-1]
        pct_change = ((forecast_end - last_actual) / last_actual) * 100
        if pct_change > 5:
            direction = f'UP {pct_change:.1f}%'
        elif pct_change < -5:
            direction = f'DOWN {abs(pct_change):.1f}%'
        else:
            direction = f'STABLE ({pct_change:.1f}%)'
        forecast_directions.append(direction)
    else:
        forecast_directions.append('N/A')

ciip_df['12M_Forecast'] = forecast_directions

# Add procurement action
def procurement_action(row):
    if row['Risk_Level'] == 'CRITICAL':
        return 'IMMEDIATE: Review safety stock, activate alternate sourcing'
    elif row['Risk_Level'] == 'HIGH':
        return 'URGENT: Monitor weekly, prepare contingency plan'
    elif row['Risk_Level'] == 'MEDIUM':
        return 'WATCH: Monthly review, document exposure'
    else:
        return 'STABLE: Standard procurement cycle'

ciip_df['Procurement_Action'] = ciip_df.apply(procurement_action, axis=1)

print("=" * 80)
print("INDIA PHARMA SUPPLY CHAIN INTELLIGENCE SYSTEM")
print("CIIP Index Scorecard — Upstream Risk Assessment")
print("Developed by Aditya V Sivaram Poduri | India Supply Chain Signals")
print("=" * 80)
print()
print(ciip_df[['Material', 'CIIP_Score', 'Risk_Level', '12M_Forecast', 'Procurement_Action']].to_string(index=False))
print()
print("Score Guide: CRITICAL >= 8.0 | HIGH >= 6.5 | MEDIUM >= 5.0 | LOW < 5.0")

In [ ]:
# --- CIIP SCORECARD VISUALIZATION (FIXED) ---

fig, ax = plt.subplots(figsize=(14, 6))
ax.axis('off')

fig.patch.set_facecolor('#0a0a0a')
ax.set_facecolor('#0a0a0a')

fig.suptitle('CIIP INDEX SCORECARD — India Pharma Supply Chain Intelligence',
             fontsize=13, fontweight='bold', color='white', y=0.98)

colors_risk = {
    'CRITICAL': '#c00000',
    'HIGH': '#D4750A',
    'MEDIUM': '#f0c040',
    'LOW': '#70ad47'
}

bar_colors = [colors_risk[r] for r in ciip_df['Risk_Level']]

bars = ax.barh(ciip_df['Material'], ciip_df['CIIP_Score'],
               color=bar_colors, height=0.5, alpha=0.9)

ax.set_xlim(0, 13)
ax.set_xlabel('CIIP Score (0-10)', color='white', fontsize=11)
ax.tick_params(colors='white')
ax.spines['bottom'].set_color('#444440')
ax.spines['left'].set_color('#444440')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# FIXED: use row index correctly
for i, (bar, (_, row)) in enumerate(zip(bars, ciip_df.iterrows())):
    width = bar.get_width()
    label = f"{row['CIIP_Score']} — {row['Risk_Level']} | {row['12M_Forecast']}"
    ax.text(width + 0.15, bar.get_y() + bar.get_height()/2,
            label,
            va='center', ha='left', color='white', fontsize=9)

import matplotlib.patches as mpatches
legend_patches = [
    mpatches.Patch(color='#c00000', label='CRITICAL (>=8.0)'),
    mpatches.Patch(color='#D4750A', label='HIGH (>=6.5)'),
    mpatches.Patch(color='#f0c040', label='MEDIUM (>=5.0)'),
    mpatches.Patch(color='#70ad47', label='LOW (<5.0)')
]
ax.legend(handles=legend_patches, loc='lower right',
          facecolor='#141414', edgecolor='#444440',
          labelcolor='white', fontsize=9)

ax.axvline(x=8.0, color='#c00000', linestyle='--', alpha=0.4, linewidth=0.8)
ax.axvline(x=6.5, color='#D4750A', linestyle='--', alpha=0.4, linewidth=0.8)

ax.text(0.01, -0.12,
        'Aditya V Sivaram Poduri | India Supply Chain Signals | indiasupplychainsignals.substack.com',
        transform=ax.transAxes, fontsize=8, color='#888880', ha='left')

plt.tight_layout()
plt.savefig('CIIP_Scorecard.png', dpi=150, bbox_inches='tight',
            facecolor='#0a0a0a')
plt.tight_layout()
plt.show()
print("CIIP Scorecard saved.")

In [ ]:
# CELL 9 - METHODOLOGY, ASSUMPTIONS AND LIMITATIONS
import json

METHODOLOGY = {
    'Framework': 'CIIP Index - Cross-Industry Input Pressure Index',
    'Developer': 'Aditya V Sivaram Poduri',
    'Version': '2.0 - June 2026',
    'Theoretical_Basis': (
        'Leontief Input-Output Economics (Nobel Prize 1973). '
        'Industries share upstream inputs. Pressure in one sector '
        'propagates into others 6-8 weeks before appearing in procurement data.'
    ),
    'Scoring_Weights': {
        'Pharma Demand Pressure': '30% - direct category exposure',
        'Cross-Industry Competition': '45% - primary source of invisible risk',
        'Geopolitical Supply Risk': '25% - supply corridor vulnerability'
    },
    'Assumptions': [
        'Historical patterns contain 12-month predictive signal',
        'CIIP input scores reflect current 2026 market structure',
        'LNG Japan is valid proxy for Chinese industrial energy costs',
        '6-8 week upstream transmission lag holds under normal conditions'
    ],
    'Limitations': [
        'CRITICAL - Three datasets are reconstructed proxies: Soda Ash (FRED), Acetic Acid (Chemanalyst), Polysilicon (PVInsights)',
        'CRITICAL - CIIP input scores are expert-calibrated not regression-derived',
        'HIGH - Prophet accuracy degrades beyond 6-month horizon for volatile materials',
        'HIGH - Polysilicon structural break in 2022 - treat as LOW CONFIDENCE',
        'MODERATE - Model does not capture black swan events',
        'DISCLOSURE - Some datasets use reconstructed proxies due to limited access to ICIS and S&P Global. Treat PROXY materials as directional indicators only.'
    ],
    'Data_Sources': {
        'Soda Ash':    'FRED Reconstructed Proxy | PROXY | 2015-2025',
        'Methanol':    'Methanex Contract Prices  | PRIMARY | 2015-2025',
        'Acetic Acid': 'Chemanalyst/ICIS Proxy    | PROXY | 2015-2025',
        'LNG Japan':   'World Bank Pink Sheet     | PRIMARY | 2015-2025',
        'Polysilicon': 'PVInsights/Bloomberg Proxy | PROXY | 2015-2025'
    }
}

print('=' * 75)
print('CIIP INDEX v2.0 - METHODOLOGY AND LIMITATIONS DISCLOSURE')
print('=' * 75)
print(f'Framework: {METHODOLOGY["Framework"]}')
print(f'Version:   {METHODOLOGY["Version"]}')
print(f'Basis: {METHODOLOGY["Theoretical_Basis"]}')
print('\nSCORING WEIGHTS:')
for k,v in METHODOLOGY['Scoring_Weights'].items():
    print(f'  {k}: {v}')
print('\nASSUMPTIONS:')
for i,a in enumerate(METHODOLOGY['Assumptions'],1):
    print(f'  {i}. {a}')
print('\nLIMITATIONS:')
for i,l in enumerate(METHODOLOGY['Limitations'],1):
    print(f'  {i}. {l}')
print('\nDATA SOURCES:')
for mat,src in METHODOLOGY['Data_Sources'].items():
    print(f'  {mat:<15} {src}')
with open('ciip_methodology_v2.json','w') as f:
    json.dump(METHODOLOGY, f, indent=2)
print('\nMethodology saved to ciip_methodology_v2.json')


In [ ]:
# ============================================================
# PHASE 1 — CELL 1 (REVISED): RIGOROUS FORECAST VALIDATION
# MAPE, MAE, RMSE, Train/Test Split, Volatility Analysis
# ============================================================

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Dataset credibility ratings — transparently disclosed
# Reflects source quality, reconstruction extent, and market representativeness
DATASET_CREDIBILITY = {
    'Soda Ash':    {'rating': 'PROXY',    'score': 0.65, 'note': 'FRED reconstructed proxy — not primary market data'},
    'Methanol':    {'rating': 'PRIMARY',  'score': 0.95, 'note': 'Methanex contract prices — highly reliable source'},
    'Acetic Acid': {'rating': 'PROXY',    'score': 0.60, 'note': 'Chemanalyst/ICIS trend proxy — partial reconstruction'},
    'LNG Japan':   {'rating': 'PRIMARY',  'score': 0.92, 'note': 'World Bank Pink Sheet — authoritative source'},
    'Polysilicon': {'rating': 'PROXY',    'score': 0.55, 'note': 'PVInsights/Bloomberg proxy — structural break 2022'},
}

def calculate_volatility_metrics(price_series):
    """
    Calculate price volatility metrics.
    High volatility = lower forecast reliability.
    """
    returns = price_series.pct_change().dropna()

    # Coefficient of variation — normalised volatility measure
    cv = (price_series.std() / price_series.mean()) * 100

    # Annualised volatility (monthly data, sqrt(12))
    annualised_vol = returns.std() * np.sqrt(12) * 100

    # Maximum drawdown
    rolling_max = price_series.cummax()
    drawdown = (price_series - rolling_max) / rolling_max
    max_drawdown = abs(drawdown.min()) * 100

    # Structural break detection — large price jumps
    monthly_changes = abs(returns)
    extreme_moves = (monthly_changes > 0.15).sum()  # >15% monthly change

    # Volatility tier
    if cv < 15:
        vol_tier = 'LOW'
        vol_penalty = 0.0
    elif cv < 30:
        vol_tier = 'MODERATE'
        vol_penalty = 0.1
    elif cv < 50:
        vol_tier = 'HIGH'
        vol_penalty = 0.25
    else:
        vol_tier = 'EXTREME'
        vol_penalty = 0.40

    return {
        'CV_Pct': round(cv, 1),
        'Annualised_Vol_Pct': round(annualised_vol, 1),
        'Max_Drawdown_Pct': round(max_drawdown, 1),
        'Extreme_Moves_Count': int(extreme_moves),
        'Volatility_Tier': vol_tier,
        'Volatility_Penalty': vol_penalty
    }

def calculate_forecast_accuracy_rigorous(material, forecast_data, credibility_data):
    """
    Rigorous forecast validation with train/test split.
    Test set: last 24 months (not 12) for more robust validation.
    """
    actual = forecast_data['actual']['y'].reset_index(drop=True)
    forecast_df = forecast_data['forecast']
    actual_dates = forecast_data['actual']['ds']

    # Align forecast to actual dates
    forecast_aligned = forecast_df[
        forecast_df['ds'].isin(actual_dates)
    ]['yhat'].reset_index(drop=True)

    min_len = min(len(actual), len(forecast_aligned))
    actual = actual[:min_len]
    forecast_aligned = forecast_aligned[:min_len]

    # Train/test split — last 24 months as test
    n_test = min(24, int(len(actual) * 0.20))

    actual_train = actual[:-n_test]
    actual_test = actual[-n_test:]
    forecast_test = forecast_aligned[-n_test:]

    # Core accuracy metrics
    mape = np.mean(
        np.abs((actual_test.values - forecast_test.values) /
               np.where(actual_test.values == 0, 1, actual_test.values))
    ) * 100

    mae = mean_absolute_error(actual_test.values, forecast_test.values)
    rmse = np.sqrt(mean_squared_error(actual_test.values, forecast_test.values))

    # Directional accuracy
    actual_dir = np.sign(np.diff(actual_test.values))
    forecast_dir = np.sign(np.diff(forecast_test.values))
    dir_accuracy = np.mean(actual_dir == forecast_dir) * 100 if len(actual_dir) > 0 else 0

    # Bias — is model systematically over or under forecasting?
    bias = np.mean(forecast_test.values - actual_test.values)
    bias_pct = (bias / np.mean(actual_test.values)) * 100

    # Forecast grade based on MAPE
    if mape < 8:
        grade = 'EXCELLENT'
    elif mape < 15:
        grade = 'GOOD'
    elif mape < 25:
        grade = 'ACCEPTABLE'
    else:
        grade = 'POOR'

    # Volatility metrics
    vol_metrics = calculate_volatility_metrics(actual)

    # Dataset credibility
    cred = credibility_data[material]

    return {
        'Material': material,
        'Train_Months': len(actual_train),
        'Test_Months': n_test,
        'MAPE_Pct': round(mape, 2),
        'MAE': round(mae, 2),
        'RMSE': round(rmse, 2),
        'Directional_Accuracy_Pct': round(dir_accuracy, 1),
        'Bias_Pct': round(bias_pct, 2),
        'Forecast_Grade': grade,
        'Volatility_Tier': vol_metrics['Volatility_Tier'],
        'CV_Pct': vol_metrics['CV_Pct'],
        'Annualised_Vol_Pct': vol_metrics['Annualised_Vol_Pct'],
        'Extreme_Moves': vol_metrics['Extreme_Moves_Count'],
        'Volatility_Penalty': vol_metrics['Volatility_Penalty'],
        'Dataset_Rating': cred['rating'],
        'Dataset_Score': cred['score'],
        'Dataset_Note': cred['note']
    }

# Run rigorous validation for all five materials
print("Running rigorous forecast validation...")
print("Train/test split: 80/20 (minimum 24-month test window)")
print()

accuracy_results = []
for material, data in forecasts.items():
    result = calculate_forecast_accuracy_rigorous(material, data, DATASET_CREDIBILITY)
    accuracy_results.append(result)

accuracy_df = pd.DataFrame(accuracy_results)

print("=" * 90)
print("FORECAST ACCURACY — Rigorous Validation with Train/Test Split")
print("=" * 90)
print(accuracy_df[[
    'Material', 'Train_Months', 'Test_Months',
    'MAPE_Pct', 'MAE', 'RMSE',
    'Directional_Accuracy_Pct', 'Bias_Pct', 'Forecast_Grade'
]].to_string(index=False))

print()
print("=" * 90)
print("VOLATILITY AND DATASET CREDIBILITY ASSESSMENT")
print("=" * 90)
print(accuracy_df[[
    'Material', 'Volatility_Tier', 'CV_Pct',
    'Extreme_Moves', 'Dataset_Rating', 'Dataset_Note'
]].to_string(index=False))

print()
print("DATASET CREDIBILITY DISCLOSURE:")
print("Some datasets include reconstructed public-market proxies due to limited")
print("access to commercial commodity databases (ICIS, S&P Global Commodity Insights).")
print("PRIMARY rated sources are directly sourced from authoritative publishers.")
print("PROXY rated sources carry additional uncertainty and should be treated as")
print("directional indicators rather than precise price references.")

In [ ]:
# ============================================================
# PHASE 1 — CELL 2 (REVISED): MULTI-FACTOR CONFIDENCE SCORING
# Volatility + Dataset Quality + Model Error + Signal Consistency
# ============================================================

# Import dependency scores — how dependent is India on imports for each material
# Scale 1-3: 1=mostly domestic, 2=mixed, 3=heavily imported
IMPORT_DEPENDENCY = {
    'Soda Ash':    {'score': 2, 'note': '20% imported, 80% domestic synthetic'},
    'Methanol':    {'score': 3, 'note': 'Heavily imported, limited domestic capacity'},
    'Acetic Acid': {'score': 3, 'note': 'Majority imported from China and SE Asia'},
    'LNG Japan':   {'score': 3, 'note': 'India has no LNG production — 100% import'},
    'Polysilicon': {'score': 3, 'note': 'Zero domestic production — 100% China import'},
}

# Supplier concentration scores — how concentrated is the supply source
# Scale 1-3: 1=diversified, 2=moderate concentration, 3=single/dual source
SUPPLIER_CONCENTRATION = {
    'Soda Ash':    {'score': 2, 'note': 'Turkey, US, Kenya — moderate diversification'},
    'Methanol':    {'score': 2, 'note': 'Middle East, Trinidad, China — moderate'},
    'Acetic Acid': {'score': 3, 'note': 'China dominant — high concentration risk'},
    'LNG Japan':   {'score': 2, 'note': 'Qatar, Australia, US — moderate'},
    'Polysilicon': {'score': 3, 'note': 'China controls 97% — extreme concentration'},
}

def calculate_multi_factor_confidence(material, accuracy_row):
    """
    Multi-factor confidence scoring with transparent component breakdown.

    Four weighted components:
    1. Model Accuracy (30%) — MAPE-based, penalised for volatility
    2. Dataset Credibility (25%) — source quality and reconstruction extent
    3. Signal Stability (25%) — volatility-adjusted reliability
    4. Supply Structure Clarity (20%) — import dependency + concentration

    Returns HIGH / MODERATE / LOW / INDICATIVE ONLY
    with full component breakdown for transparency.
    """

    # COMPONENT 1: Model Accuracy Score (0-1)
    mape = accuracy_row['MAPE_Pct']
    vol_penalty = accuracy_row['Volatility_Penalty']

    if mape < 8:
        base_accuracy = 1.0
    elif mape < 15:
        base_accuracy = 0.80
    elif mape < 25:
        base_accuracy = 0.60
    else:
        base_accuracy = 0.35

    # Apply volatility penalty — high volatility reduces model confidence
    accuracy_score = base_accuracy * (1 - vol_penalty)

    # COMPONENT 2: Dataset Credibility Score (0-1)
    dataset_score = DATASET_CREDIBILITY[material]['score']

    # COMPONENT 3: Signal Stability Score (0-1)
    cv = accuracy_row['CV_Pct']
    extreme_moves = accuracy_row['Extreme_Moves']
    dir_acc = accuracy_row['Directional_Accuracy_Pct']

    # Penalise for extreme price moves (structural breaks)
    stability_base = dir_acc / 100
    structural_break_penalty = min(0.4, extreme_moves * 0.05)
    signal_stability = stability_base * (1 - structural_break_penalty)

    # COMPONENT 4: Supply Structure Clarity (0-1)
    imp_dep = IMPORT_DEPENDENCY[material]['score']
    sup_conc = SUPPLIER_CONCENTRATION[material]['score']
    # Higher import dependency + higher concentration = clearer risk signal
    # (we can be MORE confident the risk is real, even if less able to hedge)
    supply_clarity = ((imp_dep + sup_conc) / 6)

    # Weighted composite confidence
    composite = (accuracy_score * 0.30 +
                 dataset_score * 0.25 +
                 signal_stability * 0.25 +
                 supply_clarity * 0.20)

    confidence_pct = round(composite * 100, 1)

    # Confidence tier
    if confidence_pct >= 78:
        tier = 'HIGH'
        tier_note = 'Assessment is analytically robust'
    elif confidence_pct >= 62:
        tier = 'MODERATE'
        tier_note = 'Assessment is directionally reliable'
    elif confidence_pct >= 48:
        tier = 'LOW'
        tier_note = 'Treat as indicative signal only'
    else:
        tier = 'INDICATIVE ONLY'
        tier_note = 'High uncertainty — supplementary research required'

    return {
        'Material': material,
        'Confidence_Score': confidence_pct,
        'Confidence_Tier': tier,
        'Tier_Note': tier_note,
        'Model_Accuracy_Component': round(accuracy_score * 100, 1),
        'Dataset_Credibility_Component': round(dataset_score * 100, 1),
        'Signal_Stability_Component': round(signal_stability * 100, 1),
        'Supply_Structure_Component': round(supply_clarity * 100, 1),
        'Import_Dependency': IMPORT_DEPENDENCY[material]['note'],
        'Supplier_Concentration': SUPPLIER_CONCENTRATION[material]['note'],
        'Volatility_Penalty_Applied': f"{accuracy_row['Volatility_Penalty']*100:.0f}%"
    }

# Calculate multi-factor confidence for all materials
confidence_results = []
for _, row in accuracy_df.iterrows():
    conf = calculate_multi_factor_confidence(row['Material'], row)
    confidence_results.append(conf)

confidence_df = pd.DataFrame(confidence_results)

print("=" * 90)
print("MULTI-FACTOR CONFIDENCE SCORING — Component Breakdown")
print("=" * 90)
print(confidence_df[[
    'Material', 'Confidence_Score', 'Confidence_Tier',
    'Model_Accuracy_Component', 'Dataset_Credibility_Component',
    'Signal_Stability_Component', 'Supply_Structure_Component'
]].to_string(index=False))

print()
print("CONFIDENCE TIER DEFINITIONS:")
print("HIGH (>=78%)      — Analytically robust. Use for procurement decisions.")
print("MODERATE (>=62%)  — Directionally reliable. Use with corroborating signals.")
print("LOW (>=48%)       — Indicative only. Supplement with primary market research.")
print("INDICATIVE ONLY   — High uncertainty. Not suitable for standalone decisions.")

print()
print("IMPORT DEPENDENCY AND SUPPLIER CONCENTRATION:")
for _, row in confidence_df.iterrows():
    print(f"  {row['Material']:<15} | Import: {row['Import_Dependency']}")
    print(f"                  | Concentration: {row['Supplier_Concentration']}")
    print(f"                  | Volatility Penalty: {row['Volatility_Penalty_Applied']}")
    print()

# Merge into ciip_df
ciip_df = ciip_df.drop(columns=[c for c in ciip_df.columns
                                  if c in ['Confidence_Score', 'Confidence_Tier']],
                         errors='ignore')
ciip_df = ciip_df.merge(
    confidence_df[['Material', 'Confidence_Score', 'Confidence_Tier', 'Tier_Note']],
    on='Material'
)

print("FINAL CIIP SCORECARD WITH CONFIDENCE:")
print(ciip_df[[
    'Material', 'CIIP_Score', 'Risk_Level',
    'Confidence_Score', 'Confidence_Tier'
]].to_string(index=False))

In [ ]:
# ============================================================
# PHASE 1 — CELL 3 (NEW): SCENARIO INTELLIGENCE ENGINE
# What-if analysis, stress testing, shock modelling
# ============================================================

def run_scenario(scenario_name, shocks, base_ciip_df, forecasts_dict):
    """
    Scenario engine: apply price shocks to forecast outputs
    and recalculate downstream procurement impact.

    shocks: dict of {material: pct_change}
    e.g. {'LNG Japan': 0.40} = LNG rises 40%
    """
    results = []

    for material in base_ciip_df['Material']:
        base_row = base_ciip_df[base_ciip_df['Material'] == material].iloc[0]
        base_forecast = forecasts_dict[material]['forecast']['yhat'].iloc[-1]
        last_actual = forecasts_dict[material]['actual']['y'].iloc[-1]

        # Apply shock if defined for this material
        shock_pct = shocks.get(material, 0)
        shocked_price = base_forecast * (1 + shock_pct)
        price_change_from_actual = ((shocked_price - last_actual) / last_actual) * 100

        # Recalculate CIIP score with geopolitical amplification
        # Shocks amplify geopolitical component
        base_geo = base_row['Geopolitical']
        if abs(shock_pct) > 0.20:
            amplified_geo = min(3, base_geo * 1.3)
        elif abs(shock_pct) > 0.10:
            amplified_geo = min(3, base_geo * 1.15)
        else:
            amplified_geo = base_geo

        # Recalculate CIIP
        cross_industry = (base_row['Solar_Demand'] +
                         base_row['Construction'] +
                         base_row['FMCG_Demand']) / 3
        raw = (base_row['Pharma_Demand'] * 0.30 +
               cross_industry * 0.45 +
               amplified_geo * 0.25)
        new_ciip = round(raw * (10/3), 2)

        # Risk level change
        old_risk = base_row['Risk_Level']
        if new_ciip >= 8.0:
            new_risk = 'CRITICAL'
        elif new_ciip >= 6.5:
            new_risk = 'HIGH'
        elif new_ciip >= 5.0:
            new_risk = 'MEDIUM'
        else:
            new_risk = 'LOW'

        risk_escalated = new_risk != old_risk and \
                        ['LOW','MEDIUM','HIGH','CRITICAL'].index(new_risk) > \
                        ['LOW','MEDIUM','HIGH','CRITICAL'].index(old_risk)

        # Procurement cost impact (approximate)
        # Assumes 10-15% of pharma input cost is this material
        cost_impact_pct = price_change_from_actual * 0.12

        results.append({
            'Material': material,
            'Base_Forecast': round(base_forecast, 2),
            'Shocked_Price': round(shocked_price, 2),
            'Shock_Applied_Pct': f"{shock_pct*100:+.0f}%",
            'Price_vs_Actual_Pct': round(price_change_from_actual, 1),
            'Base_CIIP': base_row['CIIP_Score'],
            'Scenario_CIIP': new_ciip,
            'CIIP_Delta': round(new_ciip - base_row['CIIP_Score'], 2),
            'Base_Risk': old_risk,
            'Scenario_Risk': new_risk,
            'Risk_Escalated': risk_escalated,
            'Est_Cost_Impact_Pct': round(cost_impact_pct, 2)
        })

    scenario_df = pd.DataFrame(results)
    return scenario_df

# Define three enterprise scenarios
SCENARIOS = {
    'Scenario 1 — LNG Surge (Hormuz Escalation)': {
        'LNG Japan': 0.40,
        'Methanol': 0.15,
        'Acetic Acid': 0.10
    },
    'Scenario 2 — China Export Controls': {
        'Polysilicon': -0.30,
        'Acetic Acid': 0.25,
        'Methanol': 0.20
    },
    'Scenario 3 — Solar Demand Surge (India PLI Acceleration)': {
        'Soda Ash': 0.20,
        'Polysilicon': 0.15,
        'LNG Japan': 0.08
    }
}

print("=" * 90)
print("SCENARIO INTELLIGENCE ENGINE — Stress Testing and Shock Modelling")
print("=" * 90)

scenario_outputs = {}

for scenario_name, shocks in SCENARIOS.items():
    print(f"\n{'='*90}")
    print(f"SCENARIO: {scenario_name}")
    print(f"Shocks applied: {shocks}")
    print()

    scenario_df = run_scenario(scenario_name, shocks, ciip_df, forecasts)
    scenario_outputs[scenario_name] = scenario_df

    # Print summary
    print(scenario_df[[
        'Material', 'Shock_Applied_Pct', 'Price_vs_Actual_Pct',
        'Base_CIIP', 'Scenario_CIIP', 'CIIP_Delta',
        'Base_Risk', 'Scenario_Risk', 'Risk_Escalated'
    ]].to_string(index=False))

    # Highlight escalations
    escalations = scenario_df[scenario_df['Risk_Escalated'] == True]
    if len(escalations) > 0:
        print(f"\nALERT: {len(escalations)} material(s) escalate to higher risk tier:")
        for _, row in escalations.iterrows():
            print(f"  {row['Material']}: {row['Base_Risk']} -> {row['Scenario_Risk']} (CIIP: {row['Base_CIIP']} -> {row['Scenario_CIIP']})")
    else:
        print("\nNo risk tier escalations in this scenario.")

print(f"\n{'='*90}")
print("SCENARIO COMPARISON SUMMARY")
print(f"{'='*90}")

# Average CIIP impact per scenario
for scenario_name, df in scenario_outputs.items():
    avg_delta = df['CIIP_Delta'].mean()
    escalations = df['Risk_Escalated'].sum()
    max_impact = df['CIIP_Delta'].max()
    print(f"\n{scenario_name}")
    print(f"  Avg CIIP delta: {avg_delta:+.2f} | Escalations: {escalations} | Max impact: {max_impact:+.2f}")

In [ ]:
# ============================================================
# PHASE 1 — CELL 4 (NEW): MONTE CARLO SIMULATION
# Probabilistic risk assessment with 10,000 simulations
# ============================================================

np.random.seed(42)
N_SIMULATIONS = 10000

def run_monte_carlo(material, forecast_data, accuracy_row, n_sim=10000):
    """
    Monte Carlo simulation for CIIP score uncertainty quantification.

    Simulates price paths based on:
    - Prophet forecast as mean
    - Historical volatility as standard deviation
    - Structural break probability from extreme moves history
    """
    base_forecast = forecast_data['forecast']['yhat'].iloc[-1]
    last_actual = forecast_data['actual']['y'].iloc[-1]

    # Annualised volatility from accuracy metrics
    annual_vol = accuracy_row['Annualised_Vol_Pct'] / 100
    monthly_vol = annual_vol / np.sqrt(12)

    # Simulate 12-month price paths
    simulated_final_prices = []

    for _ in range(n_sim):
        price = last_actual
        for month in range(12):
            # Random monthly return with fat tails (t-distribution)
            shock = np.random.standard_t(df=5) * monthly_vol
            price = price * (1 + shock)
        simulated_final_prices.append(price)

    simulated_prices = np.array(simulated_final_prices)

    # Price distribution statistics
    p5 = np.percentile(simulated_prices, 5)
    p25 = np.percentile(simulated_prices, 25)
    p50 = np.percentile(simulated_prices, 50)
    p75 = np.percentile(simulated_prices, 75)
    p95 = np.percentile(simulated_prices, 95)

    # Probability of significant price increase (>20%)
    prob_surge = (simulated_prices > last_actual * 1.20).mean() * 100

    # Probability of price decline (>10%)
    prob_decline = (simulated_prices < last_actual * 0.90).mean() * 100

    # Value at Risk (95th percentile upside cost exposure)
    var_95 = ((p95 - last_actual) / last_actual) * 100

    return {
        'Material': material,
        'Current_Price': round(last_actual, 2),
        'Prophet_Forecast': round(base_forecast, 2),
        'MC_P5': round(p5, 2),
        'MC_P25': round(p25, 2),
        'MC_Median': round(p50, 2),
        'MC_P75': round(p75, 2),
        'MC_P95': round(p95, 2),
        'Prob_Surge_20pct': round(prob_surge, 1),
        'Prob_Decline_10pct': round(prob_decline, 1),
        'VaR_95_Pct': round(var_95, 1),
        'Simulations': n_sim
    }

print("Running Monte Carlo simulation (10,000 paths per material)...")
print()

mc_results = []
for material, data in forecasts.items():
    acc_row = accuracy_df[accuracy_df['Material'] == material].iloc[0]
    mc = run_monte_carlo(material, data, acc_row, N_SIMULATIONS)
    mc_results.append(mc)

mc_df = pd.DataFrame(mc_results)

print("=" * 90)
print(f"MONTE CARLO SIMULATION — {N_SIMULATIONS:,} Paths Per Material (12-Month Horizon)")
print("=" * 90)
print(mc_df[[
    'Material', 'Current_Price', 'MC_P5', 'MC_P25',
    'MC_Median', 'MC_P75', 'MC_P95'
]].to_string(index=False))

print()
print("=" * 90)
print("RISK PROBABILITY ASSESSMENT")
print("=" * 90)
print(mc_df[[
    'Material', 'Prob_Surge_20pct', 'Prob_Decline_10pct', 'VaR_95_Pct'
]].to_string(index=False))

print()
print("INTERPRETATION:")
print("Prob_Surge_20pct  — Probability price rises >20% over 12 months")
print("Prob_Decline_10pct — Probability price falls >10% over 12 months")
print("VaR_95_Pct        — Worst-case price increase at 95th percentile")

# Merge MC results into main analysis
ciip_df = ciip_df.merge(
    mc_df[['Material', 'Prob_Surge_20pct', 'VaR_95_Pct', 'MC_P95']],
    on='Material'
)

print()
print("FINAL INTEGRATED SCORECARD:")
print(ciip_df[[
    'Material', 'CIIP_Score', 'Risk_Level',
    'Confidence_Tier', 'Prob_Surge_20pct', 'VaR_95_Pct'
]].to_string(index=False))

In [ ]:
# ============================================================
# PHASE 1 — CELL 5 (NEW): ANALYTICAL WEIGHTING FRAMEWORK
# Correlation-based + Import Dependency + Supplier Concentration
# ============================================================

# Energy correlation matrix — how correlated are these materials to LNG/energy
# Based on economic logic and commodity market relationships
# Scale: 0 to 1
ENERGY_CORRELATION = {
    'Soda Ash':    0.45,  # Moderate — energy-intensive production
    'Methanol':    0.85,  # High — direct natural gas derivative
    'Acetic Acid': 0.72,  # High — methanol-dependent production
    'LNG Japan':   1.00,  # Perfect — IS the energy measure
    'Polysilicon': 0.68,  # High — energy-intensive silicon purification
}

def calculate_analytical_weights(material, ciip_row, confidence_row, mc_row):
    """
    Four-component analytical weighting framework.

    This replaces heuristic CIIP scoring with analytically-grounded weights.

    Components:
    1. Volatility weight — higher volatility = higher weight in risk calculation
    2. Import dependency weight — higher dependency = higher exposure
    3. Energy correlation weight — higher correlation = higher geopolitical linkage
    4. Supplier concentration weight — higher concentration = higher systemic risk

    The weighted CIIP score adjusts the base CIIP for these structural factors.
    """
    base_ciip = ciip_row['CIIP_Score']

    # Component 1: Volatility weight
    vol_tier = accuracy_df[accuracy_df['Material'] == material]['Volatility_Tier'].values[0]
    vol_weight = {'LOW': 0.8, 'MODERATE': 1.0, 'HIGH': 1.2, 'EXTREME': 1.4}.get(vol_tier, 1.0)

    # Component 2: Import dependency weight
    imp_score = IMPORT_DEPENDENCY[material]['score']
    imp_weight = {1: 0.85, 2: 1.0, 3: 1.20}.get(imp_score, 1.0)

    # Component 3: Energy correlation weight
    energy_corr = ENERGY_CORRELATION[material]
    energy_weight = 0.85 + (energy_corr * 0.30)  # Range: 0.85 to 1.15

    # Component 4: Supplier concentration weight
    conc_score = SUPPLIER_CONCENTRATION[material]['score']
    conc_weight = {1: 0.85, 2: 1.0, 3: 1.25}.get(conc_score, 1.0)

    # Monte Carlo VaR adjustment
    var_95 = mc_row['VaR_95_Pct']
    var_weight = 1.0 + (max(0, var_95 - 20) / 100)  # Amplify if VaR > 20%

    # Composite weight — geometric mean to avoid excessive amplification
    composite_weight = (vol_weight * imp_weight * energy_weight *
                       conc_weight * var_weight) ** 0.2

    # Weighted CIIP — analytically adjusted
    weighted_ciip = min(10.0, round(base_ciip * composite_weight, 2))

    # Risk level from weighted score
    if weighted_ciip >= 8.0:
        weighted_risk = 'CRITICAL'
    elif weighted_ciip >= 6.5:
        weighted_risk = 'HIGH'
    elif weighted_ciip >= 5.0:
        weighted_risk = 'MEDIUM'
    else:
        weighted_risk = 'LOW'

    return {
        'Material': material,
        'Base_CIIP': base_ciip,
        'Volatility_Weight': round(vol_weight, 3),
        'Import_Dependency_Weight': round(imp_weight, 3),
        'Energy_Correlation_Weight': round(energy_weight, 3),
        'Supplier_Concentration_Weight': round(conc_weight, 3),
        'VaR_Adjustment_Weight': round(var_weight, 3),
        'Composite_Weight': round(composite_weight, 3),
        'Weighted_CIIP': weighted_ciip,
        'Weighted_Risk_Level': weighted_risk,
        'CIIP_Adjustment': round(weighted_ciip - base_ciip, 2)
    }

weights_results = []
for _, row in ciip_df.iterrows():
    material = row['Material']
    conf_row = confidence_df[confidence_df['Material'] == material].iloc[0]
    mc_row = mc_df[mc_df['Material'] == material].iloc[0]
    weights = calculate_analytical_weights(material, row, conf_row, mc_row)
    weights_results.append(weights)

weights_df = pd.DataFrame(weights_results)

print("=" * 90)
print("ANALYTICAL WEIGHTING FRAMEWORK")
print("Volatility + Import Dependency + Energy Correlation + Supplier Concentration")
print("=" * 90)
print(weights_df[[
    'Material', 'Base_CIIP',
    'Volatility_Weight', 'Import_Dependency_Weight',
    'Energy_Correlation_Weight', 'Supplier_Concentration_Weight',
    'Composite_Weight', 'Weighted_CIIP', 'CIIP_Adjustment'
]].to_string(index=False))

print()
print("=" * 90)
print("BASE CIIP vs ANALYTICALLY WEIGHTED CIIP — COMPARISON")
print("=" * 90)
comparison = weights_df[['Material', 'Base_CIIP', 'Weighted_CIIP',
                          'CIIP_Adjustment', 'Weighted_Risk_Level']].copy()
comparison['Direction'] = comparison['CIIP_Adjustment'].apply(
    lambda x: 'AMPLIFIED' if x > 0 else 'MODERATED' if x < 0 else 'UNCHANGED'
)
print(comparison.to_string(index=False))

# Update ciip_df with weighted scores
ciip_df = ciip_df.merge(
    weights_df[['Material', 'Weighted_CIIP', 'Weighted_Risk_Level',
                'Composite_Weight', 'CIIP_Adjustment']],
    on='Material'
)

print()
print("Analytical weighting complete.")
print("Use Weighted_CIIP for procurement decisions.")
print("Use Base_CIIP for framework benchmarking.")

In [ ]:
# CELL 15 - ENHANCED DUAL SCORECARD
# Left: Base CIIP | Right: Weighted CIIP with Confidence
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor('#0a0a0a')
colors_risk = {'CRITICAL':'#c00000','HIGH':'#D4750A','MEDIUM':'#f0c040','LOW':'#70ad47'}

# Left panel: Base CIIP
ax1 = axes[0]
ax1.set_facecolor('#0a0a0a')
bc1 = [colors_risk[r] for r in ciip_df['Risk_Level']]
b1 = ax1.barh(ciip_df['Material'], ciip_df['CIIP_Score'], color=bc1, height=0.5, alpha=0.9)
ax1.set_xlim(0,13)
ax1.set_xlabel('CIIP Score (0-10)', color='white', fontsize=11)
ax1.set_title('Base CIIP Score', color='white', fontsize=12, fontweight='bold')
ax1.tick_params(colors='white')
for sp in ax1.spines.values(): sp.set_color('#2a2a2a')
for bar, (_, row) in zip(b1, ciip_df.iterrows()):
    ax1.text(bar.get_width()+0.15, bar.get_y()+bar.get_height()/2,
             f"{row['CIIP_Score']} - {row['Risk_Level']}",
             va='center', ha='left', color='white', fontsize=9)
ax1.axvline(x=8.0, color='#c00000', linestyle='--', alpha=0.4, linewidth=0.8)
ax1.axvline(x=6.5, color='#D4750A', linestyle='--', alpha=0.4, linewidth=0.8)

# Right panel: Weighted CIIP + Confidence
ax2 = axes[1]
ax2.set_facecolor('#0a0a0a')
bc2 = [colors_risk[r] for r in ciip_df['Weighted_Risk_Level']]
b2 = ax2.barh(ciip_df['Material'], ciip_df['Weighted_CIIP'], color=bc2, height=0.5, alpha=0.9)
ax2.set_xlim(0,13)
ax2.set_xlabel('Weighted CIIP Score (0-10)', color='white', fontsize=11)
ax2.set_title('Analytically Weighted CIIP + Confidence', color='white', fontsize=12, fontweight='bold')
ax2.tick_params(colors='white')
for sp in ax2.spines.values(): sp.set_color('#2a2a2a')
for bar, (_, row) in zip(b2, ciip_df.iterrows()):
    label = f"{row['Weighted_CIIP']} - {row['Weighted_Risk_Level']} | Conf: {row['Confidence_Score']}%"
    ax2.text(bar.get_width()+0.15, bar.get_y()+bar.get_height()/2,
             label, va='center', ha='left', color='white', fontsize=8)
ax2.axvline(x=8.0, color='#c00000', linestyle='--', alpha=0.4, linewidth=0.8)
ax2.axvline(x=6.5, color='#D4750A', linestyle='--', alpha=0.4, linewidth=0.8)

patches = [mpatches.Patch(color=v, label=k) for k,v in colors_risk.items()]
ax1.legend(handles=patches, loc='lower right', facecolor='#141414',
           edgecolor='#444440', labelcolor='white', fontsize=8)
fig.suptitle('CIIP INDEX v2.0 - India Pharma Supply Chain Intelligence\nBase vs Analytically Weighted Risk Scores',
             fontsize=13, fontweight='bold', color='white', y=1.01)
ax1.text(0.0,-0.10,
         'Aditya V Sivaram Poduri | India Supply Chain Signals | indiasupplychainsignals.substack.com',
         transform=ax1.transAxes, fontsize=8, color='#888880')
plt.tight_layout()
plt.savefig('CIIP_Scorecard_v2.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()
print('CIIP_Scorecard_v2.png saved.')


In [ ]:
# ============================================================
# PHASE 1 — CELL 6 (NEW): FINAL INTEGRATED SCORECARD
# All analytical layers combined into one output
# ============================================================

print("=" * 90)
print("CIIP INDEX v2.0 — INDIA PHARMA SUPPLY CHAIN INTELLIGENCE SYSTEM")
print("Final Integrated Scorecard — All Analytical Layers")
print("Developed by Aditya V Sivaram Poduri | India Supply Chain Signals")
print("=" * 90)
print()

for _, row in ciip_df.sort_values('Weighted_CIIP', ascending=False).iterrows():
    material = row['Material']
    mc_row = mc_df[mc_df['Material'] == material].iloc[0]
    acc_row = accuracy_df[accuracy_df['Material'] == material].iloc[0]

    print(f"MATERIAL: {material}")
    print(f"  Base CIIP Score:      {row['CIIP_Score']} ({row['Risk_Level']})")
    print(f"  Weighted CIIP Score:  {row['Weighted_CIIP']} ({row['Weighted_Risk_Level']}) [adjustment: {row['CIIP_Adjustment']:+.2f}]")
    print(f"  Confidence:           {row['Confidence_Score']}% — {row['Confidence_Tier']}")
    print(f"  Forecast Grade:       {acc_row['Forecast_Grade']} (MAPE: {acc_row['MAPE_Pct']}%)")
    print(f"  Dataset:              {acc_row['Dataset_Rating']} — {acc_row['Dataset_Note']}")
    print(f"  Volatility:           {acc_row['Volatility_Tier']} (CV: {acc_row['CV_Pct']}%)")
    print(f"  Monte Carlo VaR 95:   {mc_row['VaR_95_Pct']:+.1f}% | Surge Probability: {mc_row['Prob_Surge_20pct']}%")
    print(f"  12M Forecast:         {row['12M_Forecast']}")
    print(f"  Procurement Action:   {row['Procurement_Action']}")
    print()

# Save complete analytical output
ciip_df.to_csv('CIIP_Complete_Analysis_v2.csv', index=False)
accuracy_df.to_csv('CIIP_Forecast_Accuracy.csv', index=False)
confidence_df.to_csv('CIIP_Confidence_Assessment.csv', index=False)
mc_df.to_csv('CIIP_Monte_Carlo.csv', index=False)
weights_df.to_csv('CIIP_Analytical_Weights.csv', index=False)

print("All analytical outputs saved:")
print("  CIIP_Complete_Analysis_v2.csv")
print("  CIIP_Forecast_Accuracy.csv")
print("  CIIP_Confidence_Assessment.csv")
print("  CIIP_Monte_Carlo.csv")
print("  CIIP_Analytical_Weights.csv")

In [ ]:
# CELL 17 - SAVE ALL ANALYTICAL OUTPUTS
import os

output_files = {
    'CIIP_Complete_Analysis_v2.csv':  ciip_df,
    'CIIP_Forecast_Accuracy.csv':     accuracy_df,
    'CIIP_Confidence_Assessment.csv': confidence_df,
    'CIIP_Monte_Carlo.csv':           mc_df,
    'CIIP_Analytical_Weights.csv':    weights_df,
}
print('=' * 60)
print('SAVING ALL ANALYTICAL OUTPUTS')
print('=' * 60)
for filename, df_out in output_files.items():
    df_out.to_csv(filename, index=False)
    size = os.path.getsize(filename)
    print(f'  {filename:<42} {len(df_out)} rows | {size} bytes')
print('\nAll outputs saved.')


In [ ]:
# CELL 18 - DOWNLOAD ALL PROJECT FILES
from google.colab import files
import os, time

print('=' * 65)
print('DOWNLOADING ALL PROJECT FILES')
print('=' * 65)

all_files = [
    ('Visualizations',   ['CIIP_Scorecard.png', 'CIIP_Scorecard_v2.png', 'CIIP_All_Forecasts.png']),
    ('Code Files',       ['app.py', 'ciip_methodology_v2.json']),
    ('CSV Outputs',      ['CIIP_Complete_Analysis_v2.csv','CIIP_Forecast_Accuracy.csv',
                          'CIIP_Confidence_Assessment.csv','CIIP_Monte_Carlo.csv',
                          'CIIP_Analytical_Weights.csv']),
    ('Source Data',      ['Soda_Ash_2015_2025.csv','Methanol_2015_2025.csv',
                          'Acetic_Acid_2015_2025.csv','LNG_Japan_2015_2025.csv',
                          'Polysilicon_2015_2025.csv']),
]

for group_name, file_list in all_files:
    print(f'\n{group_name}:')
    for f in file_list:
        if os.path.exists(f):
            size = os.path.getsize(f)
            files.download(f)
            print(f'  Downloaded: {f} ({size:,} bytes)')
            time.sleep(0.5)
        else:
            print(f'  MISSING: {f} - re-run the relevant cell first')

print('\n' + '=' * 65)
print('DOWNLOAD COMPLETE')
print('=' * 65)
print('\nAlso download notebook: File > Download > Download .ipynb')
print('GitHub repo: ciip-pharma-supply-chain-intelligence')
print('Streamlit deploy: share.streamlit.io > connect repo > app.py')


In [ ]:
# CELL 19 - GENERATE REQUIREMENTS.TXT FOR STREAMLIT DEPLOY
requirements = 'prophet==1.1.5\npandas>=1.4.0\nnumpy>=1.23\nplotly>=5.0\nstreamlit>=1.20\nscikit-learn>=1.0\nmatplotlib>=3.5\n'
with open('requirements.txt', 'w') as f:
    f.write(requirements)
print('requirements.txt:')
print(requirements)
from google.colab import files
files.download('requirements.txt')
print('requirements.txt downloaded.')


In [ ]:
# CELL 20 - GENERATE README.MD FOR GITHUB
readme_lines = [
    '# India Pharma Supply Chain Intelligence System',
    '## CIIP Index v2.0',
    '',
    '**Developer:** Aditya V Sivaram Poduri',
    '**Published:** India Supply Chain Signals | https://indiasupplychainsignals.substack.com',
    '',
    '## Business Problem',
    'Indian pharma faces upstream input cost volatility from cross-industry demand competition',
    'and geopolitical disruptions. Standard procurement systems miss signals forming 6-8 weeks upstream.',
    '',
    '## Architecture',
    '- Layer 1: Prophet forecasting for 5 pharma upstream inputs',
    '- Layer 2: CIIP Index cross-industry risk scoring with analytical weighting',
    '- Layer 3: MAPE/MAE/RMSE validation, Monte Carlo simulation, Scenario Engine',
    '',
    '## Key Findings (June 2026)',
    '| Material | CIIP Score | Risk Level | Confidence | 12M Forecast |',
    '|---|---|---|---|---|',
    '| LNG Japan | 9.50 | CRITICAL | HIGH | UP 35.2% |',
    '| Soda Ash | 8.67 | CRITICAL | MODERATE | DOWN 5.7% |',
    '| Acetic Acid | 7.50 | HIGH | MODERATE | STABLE 1.1% |',
    '| Methanol | 7.50 | HIGH | HIGH | UP 7.9% |',
    '| Polysilicon | 6.00 | MEDIUM | LOW | DOWN 59.6% |',
    '',
    '## Methodology',
    'CIIP Score = (Pharma Demand x 0.30) + (Cross-Industry x 0.45) + (Geopolitical x 0.25)',
    'Analytical Weight = f(Volatility, Import Dependency, Energy Correlation, Supplier Concentration)',
    '',
    '## Data Credibility',
    '| Material | Source | Rating |',
    '|---|---|---|',
    '| Methanol | Methanex Contract Prices | PRIMARY |',
    '| LNG Japan | World Bank Pink Sheet | PRIMARY |',
    '| Soda Ash | FRED Proxy | PROXY |',
    '| Acetic Acid | Chemanalyst/ICIS Proxy | PROXY |',
    '| Polysilicon | PVInsights Proxy | PROXY |',
    '',
    '## Tech Stack',
    'Python | Prophet | Pandas | NumPy | Plotly | Streamlit | scikit-learn',
    '',
    '## Contact',
    'Aditya V Sivaram Poduri | adityavsivaram@gmail.com | linkedin.com/in/adityasivaram'
]
readme = '\n'.join(readme_lines)
with open('README.md', 'w') as f:
    f.write(readme)
print(f'README.md written: {len(readme)} chars')
from google.colab import files
files.download('README.md')
print('README.md downloaded.')


In [ ]:
# CELL 21 - DEPENDENCY CHECK (run after session restart)
checks = {
    'forecasts':     'Cell 5  - Train Prophet models',
    'ciip_df':       'Cell 7  - CIIP scoring engine',
    'accuracy_df':   'Cell 10 - Forecast validation',
    'confidence_df': 'Cell 11 - Confidence scoring',
    'mc_df':         'Cell 13 - Monte Carlo',
    'weights_df':    'Cell 14 - Analytical weights',
}
print('=' * 60)
print('DEPENDENCY CHECK')
print('=' * 60)
all_ok = True
for var, src in checks.items():
    try:
        val = eval(var)
        shape = val.shape if hasattr(val, 'shape') else list(val.keys())
        print(f'  {var:<18} READY  {shape}')
    except NameError:
        print(f'  {var:<18} MISSING - re-run: {src}')
        all_ok = False
print()
print('All ready.' if all_ok else 'Fix missing variables then re-run downstream cells.')


In [ ]:
# CELL 22 - NOTEBOOK SUMMARY
print('=' * 70)
print('CIIP INDEX v2.0 - NOTEBOOK COMPLETE')
print('India Pharma Supply Chain Intelligence System')
print('Aditya V Sivaram Poduri | India Supply Chain Signals')
print('=' * 70)
summary = '''
CELL SEQUENCE:
  01 - PDF export setup
  02 - README documentation
  03 - Install dependencies
  04 - Load and validate datasets
  05 - Train all five Prophet models
  06 - Five-panel forecast charts
  07 - CIIP scoring engine (base scores)
  08 - CIIP scorecard visualization
  09 - Methodology and limitations disclosure
  10 - Rigorous forecast validation (MAPE/MAE/RMSE/Volatility)
  11 - Multi-factor confidence scoring (4 components)
  12 - Scenario intelligence engine (3 scenarios)
  13 - Monte Carlo simulation (10,000 paths)
  14 - Analytical weighting framework
  15 - Enhanced dual scorecard visualization
  16 - Final integrated scorecard
  17 - Save all outputs to CSV
  18 - Download all project files
  19 - Generate requirements.txt
  20 - Generate README.md
  21 - Dependency check
  22 - This summary

FILES GENERATED:
  app.py, requirements.txt, README.md
  CIIP_Scorecard.png, CIIP_Scorecard_v2.png, CIIP_All_Forecasts.png
  ciip_methodology_v2.json
  5x analytical CSV files
  5x source data CSV files

NEXT STEPS:
  1. Download .ipynb: File > Download > Download .ipynb
  2. Upload all files to GitHub: ciip-pharma-supply-chain-intelligence
  3. Deploy: share.streamlit.io
  4. Add GitHub URL to LinkedIn Projects
  5. Post LinkedIn 4-post announcement sequence
'''
print(summary)


In [ ]:
# CELL 23 - WRITE STREAMLIT DASHBOARD APP
app_lines = [
    'import streamlit as st',
    'import pandas as pd',
    'import numpy as np',
    'from prophet import Prophet',
    'import plotly.graph_objects as go',
    'import warnings',
    'warnings.filterwarnings("ignore")',
    '',
    'st.set_page_config(',
    '    page_title="India Pharma Supply Chain Intelligence System",',
    '    layout="wide"',
    ')',
    '',
    'st.title("India Pharma Supply Chain Intelligence System")',
    'st.caption("CIIP Index v2.0 | Aditya V Sivaram Poduri | India Supply Chain Signals")',
    'st.divider()',
    '',
    'ciip_data = {',
    '    "Material":     ["Soda Ash","Methanol","Acetic Acid","LNG Japan","Polysilicon"],',
    '    "CIIP_Score":   [8.67, 7.50, 7.50, 9.50, 6.00],',
    '    "Risk_Level":   ["CRITICAL","HIGH","HIGH","CRITICAL","MEDIUM"],',
    '    "Confidence":   ["MODERATE","HIGH","MODERATE","HIGH","LOW"],',
    '    "Forecast":     ["DOWN 5.7%","UP 7.9%","STABLE 1.1%","UP 35.2%","DOWN 59.6%"],',
    '    "Last_Price":   [362.90, 802.00, 680.00, 12.00, 9.00],',
    '    "Unit":         ["USD/MT","USD/MT","USD/MT","USD/MMBtu","USD/KG"],',
    '    "Action": [',
    '        "IMMEDIATE: Review safety stock. Activate alternate sourcing.",',
    '        "URGENT: Monitor weekly. Prepare contingency plan.",',
    '        "URGENT: Monitor weekly. Prepare contingency plan.",',
    '        "IMMEDIATE: Review working capital exposure.",',
    '        "WATCH: Monthly review. Document exposure."',
    '    ]',
    '}',
    'df = pd.DataFrame(ciip_data)',
    '',
    'col1,col2,col3 = st.columns(3)',
    'col1.metric("CRITICAL Materials", int((df["Risk_Level"]=="CRITICAL").sum()))',
    'col2.metric("HIGH Risk Materials", int((df["Risk_Level"]=="HIGH").sum()))',
    'col3.metric("Avg CIIP Score", f"{df[\"CIIP_Score\"].mean():.2f}")',
    'st.divider()',
    '',
    'tab1, tab2, tab3 = st.tabs(["Risk Scorecard","Forecasts","Methodology"])',
    '',
    'with tab1:',
    '    color_map = {"CRITICAL":"#c00000","HIGH":"#D4750A","MEDIUM":"#f0c040","LOW":"#70ad47"}',
    '    fig = go.Figure(go.Bar(',
    '        y=df["Material"], x=df["CIIP_Score"], orientation="h",',
    '        marker_color=[color_map[r] for r in df["Risk_Level"]],',
    '        text=[f"{s} - {r}" for s,r in zip(df["CIIP_Score"],df["Risk_Level"])],',
    '        textposition="outside", textfont=dict(color="white",size=11)',
    '    ))',
    '    fig.update_layout(paper_bgcolor="#0a0a0a",plot_bgcolor="#0a0a0a",',
    '                      font=dict(color="white"),xaxis=dict(range=[0,13]),',
    '                      height=350,margin=dict(r=200))',
    '    st.plotly_chart(fig, use_container_width=True)',
    '    st.dataframe(df[["Material","CIIP_Score","Risk_Level","Confidence","Forecast","Action"]],',
    '                 hide_index=True, use_container_width=True)',
    '',
    'with tab2:',
    '    files_map = {"Soda Ash":"Soda_Ash_2015_2025.csv","Methanol":"Methanol_2015_2025.csv",',
    '                 "Acetic Acid":"Acetic_Acid_2015_2025.csv",',
    '                 "LNG Japan":"LNG_Japan_2015_2025.csv","Polysilicon":"Polysilicon_2015_2025.csv"}',
    '    selected = st.selectbox("Select material:", list(files_map.keys()))',
    '    @st.cache_data',
    '    def get_fc(mat, fname):',
    '        d = pd.read_csv(fname).rename(columns={"Date":"ds","Price":"y"})',
    '        d["ds"] = pd.to_datetime(d["ds"])',
    '        m = Prophet(yearly_seasonality=True,weekly_seasonality=False,daily_seasonality=False)',
    '        m.fit(d)',
    '        return d, m.predict(m.make_future_dataframe(periods=12,freq="MS"))',
    '    act, fore = get_fc(selected, files_map[selected])',
    '    row = df[df["Material"]==selected].iloc[0]',
    '    fore_only = fore[fore["ds"]>act["ds"].max()]',
    '    fig2 = go.Figure()',
    '    fig2.add_trace(go.Scatter(x=act["ds"],y=act["y"],mode="markers",name="Actual",',
    '                              marker=dict(color="white",size=4,opacity=0.7)))',
    '    fig2.add_trace(go.Scatter(x=fore_only["ds"],y=fore_only["yhat"],',
    '                              mode="lines",name="12M Forecast",',
    '                              line=dict(color="#D4750A",width=2.5)))',
    '    fig2.update_layout(paper_bgcolor="#0a0a0a",plot_bgcolor="#0a0a0a",',
    '                       font=dict(color="white"),height=400,',
    '                       title=f"{selected} - CIIP: {row[\"CIIP_Score\"]} ({row[\"Risk_Level\"]})")',
    '    st.plotly_chart(fig2, use_container_width=True)',
    '',
    'with tab3:',
    '    st.markdown("## Methodology")',
    '    st.markdown("**Framework:** CIIP Index - Leontief Input-Output Economics")',
    '    st.markdown("**Scoring:** Pharma 30% + Cross-Industry 45% + Geopolitical 25%")',
    '    st.markdown("**Limitations:** Three datasets are reconstructed proxies (Soda Ash, Acetic Acid, Polysilicon)")',
    '    st.markdown("**Developer:** Aditya V Sivaram Poduri | indiasupplychainsignals.substack.com")',
]
app_code = '\n'.join(app_lines)
with open('app.py','w') as f:
    f.write(app_code)
with open('app.py','r') as f:
    lines = f.readlines()
print(f'app.py written: {len(lines)} lines')
print('Streamlit app ready for deployment.')
